# 라이브러리 및 데이터 전처리 부분

In [17]:
# 라이브러리 Import
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [18]:
# CSV 파일 로드
df = pd.read_csv('데이터/통합/통합_데이터.csv')

In [19]:
# 신규 개업역의 경우에는 개업일 이후에만 분석하기 위해서 실제 있는 신규 개업역에 대해서 딕셔너리 및 딕셔너리 이용 필터링 함수 선언

# 신규 개업 역 (역명: 개업일)
new_stations = {
    "관악산(서울대)": "2022-05-28",
    "당곡": "2022-05-28",
    "보라매공원": "2022-05-28",
    "보라매병원": "2022-05-28",
    "서울대벤처타운": "2022-05-28",
    "서울지방병무청": "2022-05-28",
    "서원": "2022-05-28",
    "성남": "2024-03-30",
    "암사역사공원": "2024-08-10",
    "원종": "2023-07-01"
}

# 신규 개업역의 개업일 이후 데이터만 필터링
def filter_new_station_data(df, new_stations):
    df_filtered = df.copy()
    for station, open_date in new_stations.items():
        open_date = pd.to_datetime(open_date)
        df_filtered = df_filtered[~((df_filtered['역명'] == station) & (df_filtered['일시'] < open_date))]
    return df_filtered

In [20]:
# 일시를 datetime으로 변환
df['일시'] = pd.to_datetime(df['일시'])

# CSV 데이터 정리 부분

In [ ]:
# CSV 데이터 정리

for col in df.columns:
    print(f"열 이름: {col}")
    print(f" - 데이터 타입: {df[col].dtype}")
    print(f" - 결측값 수: {df[col].isnull().sum()}")
    print(f" - 고유값 수: {df[col].nunique()}")
    print(f" - 고유값 예시: {df[col].unique()[:10]}")
    print("-" * 50)

# 실제 데이터 분석 부분

## 계절별로 각 역의 인구 유입량에 대한 설명력을 회귀 분석하고 산점도로 시각화

In [21]:
# 계절별로 각 역의 인구 유입량에 대한 설명력을 회귀 분석(R²)으로 계산하고 산점도로 시각화하는 코드

# 계절 리스트
seasons = ['봄', '여름', '가을', '겨울']

# 평균 유입량 (모든 계절 기준)
station_avg = df_filtered.groupby('역명')['인구유입량'].mean()

for season in seasons:
    results = []
    
    # 계절 더미 변수: 해당 계절이면 1, 아니면 0
    df_season = df_filtered.copy()
    df_season['계절더미'] = (df_season['계절'] == season).astype(int)
    
    stations = df_season['역명'].unique()
    
    for station in stations:
        sub = df_season[df_season['역명'] == station]
        
        if len(sub) < 30:
            continue
        
        X = sub[['계절더미']]
        y = sub['인구유입량']
        
        model = LinearRegression().fit(X, y)
        y_pred = model.predict(X)
        r2 = r2_score(y, y_pred)
        
        results.append({
            '역명': station,
            'R2': r2,
            '평균유입량': station_avg[station]
        })
    
    # DataFrame 생성
    r2_df = pd.DataFrame(results)

    # 상위 10개 역: True/False 마커 추가
    r2_df['Top10'] = r2_df['R2'].rank(ascending=False, method='min') <= 10

    # Plotly 산점도 시각화
    fig = px.scatter(
        r2_df,
        x='평균유입량',
        y='R2',
        hover_name='역명',
        color=r2_df['Top10'].map({True: 'Top 10', False: '기타'}),
        color_discrete_map={'Top 10': 'red', '기타': 'mediumseagreen'},
        title=f"{season} 설명력 (R²) vs 평균 유입량",
        labels={'평균유입량': '역 평균 유입량', 'R2': f'{season} 설명력 (R²)', 'color': '구분'},
        opacity=0.75
    )

    fig.update_traces(marker=dict(size=8))
    fig.update_layout(width=850, height=550)
    fig.show()


## 휴일 구분 별로 각 역의 인구 유입량에 대한 설명력을 회귀 분석하고 산점도로 시각화

In [23]:
# 각 역별로 휴일 여부에 따른 인구 유입량 설명력(R²)을 계산하고 산점도로 시각화하는 코드

# 1. 휴일 더미 변수 생성: 평일=0, 주말/공휴일=1
df_holiday = df_filtered.copy()
df_holiday['휴일더미'] = (df_holiday['휴일 구분'] == '주말/공휴일').astype(int)

# 2. 역별 평균 유입량
station_avg = df_holiday.groupby('역명')['인구유입량'].mean()

# 3. R² 계산
results = []

for station in df_holiday['역명'].unique():
    sub = df_holiday[df_holiday['역명'] == station]
    
    if len(sub) < 30:
        continue
    
    X = sub[['휴일더미']]
    y = sub['인구유입량']
    
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    
    results.append({
        '역명': station,
        'R2': r2,
        '평균유입량': station_avg[station]
    })

# 4. DataFrame 생성 및 상위 10 마커
r2_df = pd.DataFrame(results)
r2_df['Top10'] = r2_df['R2'].rank(ascending=False, method='min') <= 10

# 5. Plotly 시각화
fig = px.scatter(
    r2_df,
    x='평균유입량',
    y='R2',
    hover_name='역명',
    color=r2_df['Top10'].map({True: 'Top 10', False: '기타'}),
    color_discrete_map={'Top 10': 'red', '기타': 'mediumseagreen'},
    title='휴일 여부에 대한 인구 유입량 설명력 (R²) vs 평균 유입량',
    labels={'평균유입량': '역 평균 유입량', 'R2': '설명력 (R²)', 'color': '구분'},
    opacity=0.75
)

fig.update_traces(marker=dict(size=8))
fig.update_layout(width=850, height=550)
fig.show()


## 방학 구분 별로 각 역의 인구 유입량에 대한 설명력을 회귀 분석하고 산점도로 시각화

In [24]:
# 방학 여부(여름방학, 겨울방학 vs X)에 따른 인구 유입량 설명력(R²)을 계산하고 산점도로 시각화하는 코드

# 1. 방학 여부 더미 변수 생성 (여름/겨울방학: 1, X: 0)
df_vac = df_filtered.copy()
df_vac['방학더미'] = df_vac['방학 여부'].apply(lambda x: 1 if x in ['여름방학', '겨울방학'] else 0)

# 2. 역별 평균 유입량 계산
station_avg = df_vac.groupby('역명')['인구유입량'].mean()

# 3. R² 계산
results = []

for station in df_vac['역명'].unique():
    sub = df_vac[df_vac['역명'] == station]
    
    if len(sub) < 30:
        continue
    
    X = sub[['방학더미']]
    y = sub['인구유입량']
    
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    
    results.append({
        '역명': station,
        'R2': r2,
        '평균유입량': station_avg[station]
    })

# 4. DataFrame 생성 및 Top 10 마커 지정
r2_df = pd.DataFrame(results)
r2_df['Top10'] = r2_df['R2'].rank(ascending=False, method='min') <= 10

# 5. Plotly 산점도 시각화
fig = px.scatter(
    r2_df,
    x='평균유입량',
    y='R2',
    hover_name='역명',
    color=r2_df['Top10'].map({True: 'Top 10', False: '기타'}),
    color_discrete_map={'Top 10': 'red', '기타': 'mediumseagreen'},
    title='방학 여부에 따른 인구 유입량 설명력 (R²) vs 평균 유입량',
    labels={'평균유입량': '역 평균 유입량', 'R2': '설명력 (R²)', 'color': '구분'},
    opacity=0.75
)

fig.update_traces(marker=dict(size=8))
fig.update_layout(width=850, height=550)
fig.show()

## 방학 유형 별로 각 역의 인구 유입량에 대한 설명력을 회귀 분석하고 산점도로 시각화

In [26]:
# 여름방학 및 겨울방학 여부에 따른 인구 유입량 설명력(R²)을 별도로 계산하고 산점도로 시각화하는 코드

# 1. 방학 종류 리스트
vacation_types = ['여름방학', '겨울방학']

# 2. 평균 유입량 계산
station_avg = df_filtered.groupby('역명')['인구유입량'].mean()

# 3. 반복 분석
for vac in vacation_types:
    results = []
    
    df_temp = df_filtered.copy()
    
    # 해당 방학 여부만 1, 나머지 0
    df_temp['방학더미'] = (df_temp['방학 여부'] == vac).astype(int)
    
    for station in df_temp['역명'].unique():
        sub = df_temp[df_temp['역명'] == station]
        
        if len(sub) < 30:
            continue
        
        X = sub[['방학더미']]
        y = sub['인구유입량']
        
        model = LinearRegression().fit(X, y)
        y_pred = model.predict(X)
        r2 = r2_score(y, y_pred)
        
        results.append({
            '역명': station,
            'R2': r2,
            '평균유입량': station_avg[station]
        })
    
    # 4. 결과 정리
    r2_df = pd.DataFrame(results)
    r2_df['Top10'] = r2_df['R2'].rank(ascending=False, method='min') <= 10
    
    # 5. 시각화
    fig = px.scatter(
        r2_df,
        x='평균유입량',
        y='R2',
        hover_name='역명',
        color=r2_df['Top10'].map({True: 'Top 10', False: '기타'}),
        color_discrete_map={'Top 10': 'red', '기타': 'mediumseagreen'},
        title=f"{vac} 여부에 따른 인구 유입량 설명력 (R²) vs 평균 유입량",
        labels={'평균유입량': '역 평균 유입량', 'R2': f'{vac} 설명력 (R²)', 'color': '구분'},
        opacity=0.75
    )
    
    fig.update_traces(marker=dict(size=8))
    fig.update_layout(width=850, height=550)
    fig.show()



## 날씨 유형 별로 각 역의 인구 유입량에 대한 설명력을 회귀 분석하고 산점도로 시각화

In [27]:
# 각 날씨 유형별로 인구 유입량 설명력(R²)을 계산하고 산점도로 시각화하는 코드

# 1. 날씨 유형 목록
weather_types = df_filtered['날씨 유형'].unique()

# 2. 평균 유입량 계산
station_avg = df_filtered.groupby('역명')['인구유입량'].mean()

# 3. 날씨 유형별 분석
for weather in weather_types:
    results = []
    
    df_weather = df_filtered.copy()
    df_weather['날씨더미'] = (df_weather['날씨 유형'] == weather).astype(int)
    
    for station in df_weather['역명'].unique():
        sub = df_weather[df_weather['역명'] == station]
        
        if len(sub) < 30:
            continue
        
        X = sub[['날씨더미']]
        y = sub['인구유입량']
        
        model = LinearRegression().fit(X, y)
        y_pred = model.predict(X)
        r2 = r2_score(y, y_pred)
        
        results.append({
            '역명': station,
            'R2': r2,
            '평균유입량': station_avg[station]
        })
    
    r2_df = pd.DataFrame(results)
    r2_df['Top10'] = r2_df['R2'].rank(ascending=False, method='min') <= 10

    # 4. 산점도 시각화
    fig = px.scatter(
        r2_df,
        x='평균유입량',
        y='R2',
        hover_name='역명',
        color=r2_df['Top10'].map({True: 'Top 10', False: '기타'}),
        color_discrete_map={'Top 10': 'red', '기타': 'mediumseagreen'},
        title=f"{weather} 날씨 여부에 따른 인구 유입량 설명력 (R²) vs 평균 유입량",
        labels={'평균유입량': '역 평균 유입량', 'R2': '설명력 (R²)', 'color': '구분'},
        opacity=0.75
    )

    fig.update_traces(marker=dict(size=8))
    fig.update_layout(width=850, height=550)
    fig.show()